In [1]:
from opt_targeted_transfers import HybridTargetedTransfers
from data_loaders import get_dataset
from data_utils import split_data

In [2]:
# Make train and test sets
X, y, r, features = get_dataset("malawi")
d = 3
(X_train, y_train, r_train), (X_test, y_test, r_test) = split_data(X=X[:, :d], y=y, r=r, p=0.6)

In [3]:
tt = HybridTargetedTransfers(c_bar=2.15, unconditional_tolerance=0.1, conditional_tolerance=None)

In [4]:
# Fit density functions
# Note only training for 50 epochs here for an example, in practice use the default number of epochs.
tt.fit(X_train, y_train, r_train, n_epochs=50)

KNOTS:[-3.27351677 -3.27351677 -3.27351677 -3.27351677 -1.19700685 -0.84186768
 -0.31661819  0.17797269  5.095039    5.095039    5.095039    5.095039  ]
Fitting conditional densities vs glm spline method...


100%|██████████| 50/50 [00:10<00:00,  4.98it/s, loss=-0.0146, val_loss=0.0177]  

Final Theta: tensor([[-0.9873,  0.7758, -0.2372],
        [ 0.5527, -0.3248, -0.0848],
        [-0.3383,  0.4670, -0.5115],
        [-0.1545, -0.1551,  0.1368],
        [-0.5008,  0.4967,  0.3995],
        [-0.0546, -0.3039,  0.3120],
        [ 0.4300, -0.4868,  0.4648],
        [-0.2917, -0.4789, -0.0600]], dtype=torch.float64)


In [5]:
# Run optimization algorithm
# Note only trying 10 alpha values for an example, in practice use the default number of alpha values.
opt_policy = tt.run_opt(
    X_test, r_test, n_alpha=10, path="malawi_example_unconditional_tolerance=0.1.csv"
)

Alpha range: 0.0006727523319816305, 0.6727523319816305


100%|██████████| 10/10 [00:09<00:00,  1.04it/s]


In [6]:
# Query the optimal transfer policy after running the optimization algorithm
transfer = opt_policy(X_test[[0]])
transfer

{0: [(1.3699195650479332, 1.0)]}

In [7]:
# Note that tt object also stores the optimal policy
transfer = tt.opt_policy(X_test[[0]])
transfer

{0: [(1.3699195650479332, 1.0)]}

In [8]:
# Evaluate policy. 
res = tt.evaluate(X_test, y_test, r_test)
res

{'initial_poverty_rate': 0.6511447390812347,
 'initial_poverty_gap': 0.628796805717522,
 'post_transfer_poverty_gap': 0.03883007310285531,
 'post_transfer_poverty_rate': 0.12859196490297972,
 'policy_cost': 1.471735420222309,
 'method': 'hybrid',
 'unconditional_tolerance': 0.1,
 'conditional_tolerance': None,
 'd': 3,
 'nclass': None}

In [9]:
# Can try a different tolerance without re-fitting the densities!
# Note that setting a new tolerance will clear the previously computed policy,
# so we will have to re-run the optimization.
tt.set_conditional_tolerance(conditional_tolerance=0.5)
new_opt_policy = tt.run_opt(
    X_test, r_test, n_alpha=10, path="malawi_example_uncond-tol=0.1_cond-tol=0.5.csv"
)

Alpha range: 0.0006727523319816305, 0.6727523319816305


100%|██████████| 10/10 [00:09<00:00,  1.02it/s]


In [10]:
# Evaluate policy
res = tt.evaluate(X_test, y_test, r_test)
res

{'initial_poverty_rate': 0.6511447390812347,
 'initial_poverty_gap': 0.628796805717522,
 'post_transfer_poverty_gap': 0.05669043070130785,
 'post_transfer_poverty_rate': 0.14252981104455537,
 'policy_cost': 1.4854774566206135,
 'method': 'hybrid',
 'unconditional_tolerance': 0.1,
 'conditional_tolerance': 0.5,
 'd': 3,
 'nclass': None}